In [7]:
import numpy as np
import torch

dataset = np.genfromtxt("heart_failure_clinical_records_dataset.csv",
                        delimiter=",", skip_header=1)

'''
anemia, high bp, diabetes, smoking: 0 = false
sex: 0 = woman
'''

x = dataset[:, :-1]
y = dataset[:, -1]

x = (x - x.min(axis=0)) / (x.max(axis=0) - x.min(axis=0))

x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

print(x.shape, y.shape)
print(x[0].shape, x[1].shape)
print(x.shape[1])

torch.Size([299, 12]) torch.Size([299])
torch.Size([12]) torch.Size([12])
12


In [8]:
def sigmoid(z):
  return torch.sigmoid(z)

def dense(ain, W, b, activation):
  units = W.shape[1]
  aout = torch.zeros(units)
  zout = torch.zeros(units)

  for j in range(0, units):
    wj = W[:,j]
    z = torch.dot(wj, ain) + b[j]
    zout[j] = z
    aout[j] = activation(z)
  return aout, zout

def sequential(xi, W1, b1, W2, b2):
  a1, z1 = dense(xi, W1, b1, sigmoid)
  a2, z2 = dense(a1, W2, b2, sigmoid)
  return a1, z1, a2, z2



In [9]:
torch.manual_seed(1)

W1 = torch.randn(12, 4) * 0.01
W1.requires_grad = True

b1 = torch.zeros(4, requires_grad = True)

W2 = torch.randn(4, 1) * 0.01
W2.requires_grad = True
b2 = torch.zeros(1, requires_grad=True)

alpha = 0.05
epochs = 2000

for epoch in range(epochs):
    loss = torch.tensor(0.0)

    for i in range(0, 299):
        xi = x[i]
        yi = y[i]

        a1, z1, a2, z2 = sequential(xi, W1, b1, W2, b2)
        yhat = a2[0]
        loss = loss - (yi * torch.log(yhat) + (1 - yi) * torch.log(1 - yhat))

    mean_loss = loss / 299
    mean_loss.backward()

    with torch.no_grad():

        W2 -= alpha * W2.grad
        b2 -= alpha * b2.grad
        W1 -= alpha * W1.grad
        b1 -= alpha * b1.grad

        W2.grad.zero_()
        b2.grad.zero_()
        W1.grad.zero_()
        b1.grad.zero_()

    if epoch % 200 == 0:
        print("epoch-", epoch, "cost-", mean_loss.item())

epoch- 0 cost- 0.6965325474739075
epoch- 200 cost- 0.6270570158958435
epoch- 400 cost- 0.6264320611953735
epoch- 600 cost- 0.6257157921791077
epoch- 800 cost- 0.6248466372489929
epoch- 1000 cost- 0.6237596273422241
epoch- 1200 cost- 0.6223787665367126
epoch- 1400 cost- 0.6206099987030029
epoch- 1600 cost- 0.6183426380157471
epoch- 1800 cost- 0.6154456734657288
